### Implementation of Self - Attention mechanism using trainable weight matrices.
1. Converting input embedding vector matrix into Query, Key & Value trainable matrices.
2. Input diminension of these (query, key, value) matrix should be equal to the input embedding matrix.
3. Calculating attention scores for each (word/token) using the matmul operation between `Query` and `Key` Matrices.
4. Scaling attention score to get attention weights by the below formula
$$
\mathrm{Attention Matrix}(Q, K, V)
= \mathrm{softmax}\!\left(
\frac{Q K^{\top}}{\sqrt{d_k}}
\right)
$$

5. To Calcuate the context vector for each input embedding vector (value matrix)
    - Context_Vectors = Attention_Matrix * values 

In [1]:
import torch
import torch.nn as nn

In [2]:
# Input token values
input_embeddings = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # Journey
    [0.57, 0.85, 0.64], # Starts
    [0.22, 0.58, 0.33], # With
    [0.77, 0.25, 0.40], # One
    [0.05, 0.80, 0.55], # Step
])

In [3]:
d_in = input_embeddings.shape[-1] # d_in = 3
d_out = 2 # for taking an example (for higher LLM architecture == d_in)

# Initialization of W_q, W_k, and W_v matrices
torch.manual_seed(123)
W_q = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
print(f"Initialized Query weight matrix: \n {W_q}")

W_k = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
print(f"Initialized Query weight matrix: \n {W_k}")

W_v = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
print(f"Initialized Value weight matrix: \n {W_v}")

Initialized Query weight matrix: 
 Parameter containing:
tensor([[-0.1115,  0.1204],
        [-0.3696, -0.2404],
        [-1.1969,  0.2093]])
Initialized Query weight matrix: 
 Parameter containing:
tensor([[-0.9724, -0.7550],
        [ 0.3239, -0.1085],
        [ 0.2103, -0.3908]])
Initialized Value weight matrix: 
 Parameter containing:
tensor([[ 0.2350,  0.6653],
        [ 0.3528,  0.9728],
        [-0.0386, -0.8861]])


In [4]:
# Multiply each trainable matrices with input embedding vector matrix to get
# Query, Key and Value transformation matrices

query = input_embeddings @ W_q
key = input_embeddings @ W_k
value = input_embeddings @ W_v

print(f"Query matrix: \n {query}")
print(f"Key matrix: \n {key}")
print(f"Value matrix: \n {value}")

Query matrix: 
 tensor([[-1.1686,  0.2019],
        [-1.1729, -0.0048],
        [-1.1438, -0.0018],
        [-0.6339, -0.0439],
        [-0.6570,  0.1163],
        [-0.9596, -0.0712]])
Key matrix: 
 tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.5836, -0.7649],
        [ 0.3262, -0.3395]])
Value matrix: 
 tensor([[ 0.1196, -0.3566],
        [ 0.4107,  0.6274],
        [ 0.4091,  0.6390],
        [ 0.2436,  0.4182],
        [ 0.2537,  0.4010],
        [ 0.2728,  0.3242]])


In [5]:
query_2 = query[1] # word = Journey
attention_score_2 =  query_2 @ key.T

print(f"Attention score with query 'Journey': {attention_score_2}")

Attention score with query 'Journey': tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.6882, -0.3809])


In [6]:
# Calculating attention scores for each query word
attention_scores = query @ key.T
print(f"Attention scores: \n{attention_scores}")

Attention scores: 
tensor([[ 0.0740, -0.0216,  0.0126, -0.1230,  0.5276, -0.4498],
        [ 0.2172,  0.1376,  0.1730, -0.0491,  0.6882, -0.3809],
        [ 0.2098,  0.1320,  0.1665, -0.0489,  0.6689, -0.3725],
        [ 0.1458,  0.1061,  0.1254, -0.0118,  0.4035, -0.1919],
        [ 0.0397, -0.0142,  0.0050, -0.0701,  0.2945, -0.2538],
        [ 0.2240,  0.1642,  0.1935, -0.0161,  0.6145, -0.2888]])


In [7]:
# Normalizing attention scores to get attention matrix
# Before normalization, we need to scale the attention score vectors
# Why diving with dim(keys) -> To introduce stablity in values while learning phase
# Why taking sqrt(key_dims) -> To make variance of elements comparable to one
key_dims = key.shape[-1]
attention_matrix = torch.softmax(attention_scores / key_dims**0.5, dim= -1)
#dim == -1 (To make all colums sum == 1)

print(f"Attention Matrix: \n{attention_matrix}")

Attention Matrix: 
tensor([[0.1715, 0.1603, 0.1642, 0.1492, 0.2364, 0.1184],
        [0.1726, 0.1632, 0.1673, 0.1430, 0.2408, 0.1131],
        [0.1726, 0.1633, 0.1674, 0.1437, 0.2387, 0.1143],
        [0.1712, 0.1665, 0.1688, 0.1532, 0.2055, 0.1349],
        [0.1703, 0.1639, 0.1661, 0.1575, 0.2039, 0.1383],
        [0.1726, 0.1654, 0.1689, 0.1456, 0.2274, 0.1201]])


In [8]:
# Calcuating the context vector matrix
# Query: - The word taken from the text, which we are calcuting the attention.
# Key: - All the items in the word, in which the query can check for alignment.
# Value: - Transformed input embedding matrix, we need to check how much the values are scaled when calculated attention matrix.

context_vector = attention_matrix @ value

print(f'Context Vector: \n{context_vector}')

Context Vector: 
tensor([[0.2821, 0.3399],
        [0.2829, 0.3408],
        [0.2829, 0.3408],
        [0.2841, 0.3414],
        [0.2835, 0.3407],
        [0.2836, 0.3412]])


In [15]:
# Generalization of self_attention in class or module format. 
class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        # We have to initialize the nn.Module also 
        super().__init__()
        # Initialize trainable weights query, key, & value 
        self.W_q = nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
        self.W_k = nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)
        self.W_v = nn.Parameter(torch.randn(d_in, d_out), requires_grad= False)

    def forward(self, x):
        # Calculating query, key and values matrices from trainable weight matrices. 
        query = x @ self.W_q 
        keys = x @ self.W_k
        values = x @ self.W_v

        attention_matrix = torch.softmax(query @ keys.T / keys.shape[-1]**5, dim= -1)
        context_vectors = attention_matrix @ values 

        return context_vectors


In [16]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # Journey
    [0.57, 0.85, 0.64], # Starts
    [0.22, 0.58, 0.33], # With
    [0.77, 0.25, 0.40], # One
    [0.05, 0.80, 0.55], # Step
])

d_in = 3
d_out = 2

torch.manual_seed(123)
# Making object for class SelfAttentionV1
self_att_v1 = SelfAttentionV1(d_in, d_out)

context_vecs = self_att_v1.forward(inputs)

print(f"Shape of context vectors: {context_vecs.shape}\n")
print(f"Context Vectors: \n{context_vecs}")

Shape of context vectors: torch.Size([6, 2])

Context Vectors: 
tensor([[0.2848, 0.3421],
        [0.2848, 0.3421],
        [0.2848, 0.3421],
        [0.2849, 0.3421],
        [0.2849, 0.3421],
        [0.2849, 0.3421]])


In [17]:
# Generating a more stable version of self-attention module 
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias= False):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias= qkv_bias)

    def forward(self, x):
        # Calculating query, key, value matrices with optimized Linear module approach 
        query = self.W_q(x)
        key = self.W_k(x)
        values = self.W_v(x)

        attention_matrix = torch.softmax(query @ key.T / key.shape[-1]**0.5, dim= -1)

        context_vectors = attention_matrix @ values

        return context_vectors


In [18]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # Journey
    [0.57, 0.85, 0.64], # Starts
    [0.22, 0.58, 0.33], # With
    [0.77, 0.25, 0.40], # One
    [0.05, 0.80, 0.55], # Step
])

d_in = 3
d_out = 2

torch.manual_seed(123)
# Making object for class SelfAttentionV2
self_att_v1 = SelfAttentionV2(d_in, d_out)

context_vecs = self_att_v1.forward(inputs)

print(f"Shape of context vectors: {context_vecs.shape}\n")
print(f"Context Vectors: \n{context_vecs}")

Shape of context vectors: torch.Size([6, 2])

Context Vectors: 
tensor([[-0.5441, -0.0863],
        [-0.5426, -0.0896],
        [-0.5426, -0.0896],
        [-0.5401, -0.0890],
        [-0.5429, -0.0875],
        [-0.5402, -0.0896]], grad_fn=<MmBackward0>)
